In [1]:
import pandas as pd
import time

start = time.time()

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

df = pd.read_csv(r"C:\Users\fuzzy\Downloads\Crimes_-_2001_to_Present.csv") ##Need to change this to smaller dataset

df.head()

C:\Users\fuzzy\AppData\Local\Temp\ipykernel_7664\3340893785.py:9: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\fuzzy\Downloads\Crimes_-_2001_to_Present.csv") ##Need to change this to smaller dataset


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,13311263,JG503434,07/29/2022 03:39:00 AM,023XX S TROY ST,1582,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,True,False,1033,10.0,25.0,30.0,17,NaN,NaN,2022,04/18/2024 03:40:59 PM,NaN,NaN,NaN
1,13053066,JG103252,01/03/2023 04:44:00 PM,039XX W WASHINGTON BLVD,2017,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,True,False,1122,11.0,28.0,26.0,18,NaN,NaN,2023,01/20/2024 03:41:12 PM,NaN,NaN,NaN
2,12131221,JD327000,08/10/2020 09:45:00 AM,015XX N DAMEN AVE,0326,ROBBERY,AGGRAVATED VEHICULAR HIJACKING,STREET,True,False,1424,14.0,1.0,24.0,03,1162795.0,1909900.0,2020,05/17/2025 03:40:52 PM,41.908418,-87.677407,"(41.908417822, -87.67740693)"
3,11227634,JB147599,08/26/2017 10:00:00 AM,001XX W RANDOLPH ST,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL/MOTEL,False,False,122,1.0,42.0,32.0,02,NaN,NaN,2017,02/11/2018 03:57:41 PM,NaN,NaN,NaN
4,13203321,JG415333,09/06/2023 05:00:00 PM,002XX N Wells st,1320,CRIMINAL DAMAGE,TO VEHICLE,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,122,1.0,42.0,32.0,14,1174694.0,1901831.0,2023,11/04/2023 03:40:18 PM,41.886018,-87.633938,"(41.886018055, -87.633937881)"


In [2]:
df = df.drop(['Latitude', 'Longitude', 'Location', 'X Coordinate', 'Y Coordinate'], axis=1)

df = df.dropna()

In [3]:
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%m/%d/%Y %I:%M:%S %p"
)

df["Updated On"] = pd.to_datetime(
    df["Updated On"],
    format="%m/%d/%Y %I:%M:%S %p"
)

In [4]:
df["arrest_hour"] = df["Date"].dt.hour
df["arrest_day"] = df["Date"].dt.day_name()
df["arrest_month"] = df["Date"].dt.month
df["arrest_year"] = df["Date"].dt.year

df = df.drop(['Year'], axis=1)

In [5]:
df["District"] = df["District"].astype("Int64")
df["Ward"] = df["Ward"].astype("Int64")
df["Community Area"] = df["Community Area"].astype("Int64")

In [6]:
def recode_crime(x):
    if x in ["HOMICIDE", "ASSAULT", "BATTERY", "ROBBERY",
             "KIDNAPPING", "INTIMIDATION", "STALKING"]:
        return "VIOLENT CRIME"
    
    elif x in ["CRIM SEXUAL ASSAULT", "CRIMINAL SEXUAL ASSAULT",
               "SEX OFFENSE", "HUMAN TRAFFICKING"]:
        return "SEXUAL / TRAFFICKING"

    elif x in ["THEFT", "BURGLARY", "MOTOR VEHICLE THEFT",
               "CRIMINAL DAMAGE", "CRIMINAL TRESPASS",
               "DECEPTIVE PRACTICE", "ARSON"]:
        return "PROPERTY CRIME"

    elif x in ["NARCOTICS", "OTHER NARCOTIC VIOLATION"]:
        return "DRUG OFFENSE"

    elif x in ["PROSTITUTION", "GAMBLING", "LIQUOR LAW VIOLATION",
               "PUBLIC INDECENCY", "OBSCENITY", "RITUALISM",
               "PUBLIC PEACE VIOLATION"]:
        return "PUBLIC ORDER"

    elif x in ["WEAPONS VIOLATION", "CONCEALED CARRY LICENSE VIOLATION"]:
        return "WEAPONS OFFENSE"

    elif x in ["OFFENSE INVOLVING CHILDREN"]:
        return "CRIMES AGAINST CHILDREN"

    elif x in ["INTERFERENCE WITH PUBLIC OFFICER"]:
        return "LAW ENFORCEMENT RELATED"

    elif x in ["OTHER OFFENSE"]:
        return "MISCELLANEOUS CRIME"

    elif x in ["NON-CRIMINAL"]:
        return "NON-CRIMINAL"

    else:
        return "UNCLASSIFIED"

df["Primary Type"] = df["Primary Type"].apply(recode_crime)

In [7]:
df.head(15)

,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,Updated On,arrest_hour,arrest_day,arrest_month,arrest_year
0,13311263,JG503434,2022-07-29 03:39:00,023XX S TROY ST,1582,CRIMES AGAINST CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,True,False,1033,10,25,30,17,2024-04-18 15:40:59,3,Friday,7,2022
1,13053066,JG103252,2023-01-03 16:44:00,039XX W WASHINGTON BLVD,2017,DRUG OFFENSE,MANUFACTURE / DELIVER - CRACK,SIDEWALK,True,False,1122,11,28,26,18,2024-01-20 15:41:12,16,Tuesday,1,2023
2,12131221,JD327000,2020-08-10 09:45:00,015XX N DAMEN AVE,0326,VIOLENT CRIME,AGGRAVATED VEHICULAR HIJACKING,STREET,True,False,1424,14,1,24,03,2025-05-17 15:40:52,9,Monday,8,2020
3,11227634,JB147599,2017-08-26 10:00:00,001XX W RANDOLPH ST,0281,SEXUAL / TRAFFICKING,NON-AGGRAVATED,HOTEL/MOTEL,False,False,122,1,42,32,02,2018-02-11 15:57:41,10,Saturday,8,2017
4,13203321,JG415333,2023-09-06 17:00:00,002XX N Wells st,1320,PROPERTY CRIME,TO VEHICLE,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,122,1,42,32,14,2023-11-04 15:40:18,17,Wednesday,9,2023
5,13204489,JG416325,2023-09-06 11:00:00,0000X E 8TH ST,0810,PROPERTY CRIME,OVER $500,PARKING LOT / GARAGE (NON RESIDENTIAL),False,False,123,1,4,32,06,2023-11-04 15:40:18,11,Wednesday,9,2023
6,11695116,JC272771,2019-05-21 08:20:00,018XX S CALIFORNIA AVE,0620,PROPERTY CRIME,UNLAWFUL ENTRY,RESIDENCE,False,False,1023,10,25,29,05,2024-01-19 15:40:50,8,Tuesday,5,2019
7,12419690,JE295655,2021-07-07 10:30:00,132XX S GREENWOOD AVE,1544,SEXUAL / TRAFFICKING,SEXUAL EXPLOITATION OF A CHILD,RESIDENCE,False,False,533,5,10,54,17,2024-01-19 15:40:50,10,Wednesday,7,2021
8,12729745,JF279458,2022-06-14 14:47:00,035XX N CENTRAL AVE,0340,VIOLENT CRIME,ATTEMPT STRONG ARM - NO WEAPON,BANK,True,False,1633,16,30,15,03,2024-01-19 15:40:50,14,Tuesday,6,2022
9,12835559,JF406130,2022-09-21 22:00:00,004XX E 69TH ST,0910,PROPERTY CRIME,AUTOMOBILE,OTHER (SPECIFY),True,False,322,3,6,69,07,2024-01-19 15:40:50,22,Wednesday,9,2022


In [8]:
df.dtypes

ID                               int64
Case Number                     object
Date                    datetime64[ns]
Block                           object
IUCR                            object
Primary Type                    object
Description                     object
Location Description            object
Arrest                            bool
Domestic                          bool
Beat                             int64
District                         Int64
Ward                             Int64
Community Area                   Int64
FBI Code                        object
Updated On              datetime64[ns]
arrest_hour                      int32
arrest_day                      object
arrest_month                     int32
arrest_year                      int32
dtype: object

In [9]:
hourly_counts = df.groupby(["Primary Type", "arrest_hour"]) \
                .size() \
                .unstack(fill_value=0)

print(hourly_counts)

arrest_hour                  0       1      2      3      4      5      6   \
Primary Type                                                                 
CRIMES AGAINST CHILDREN   11298     850    559    453    321    382    587   
DRUG OFFENSE              23625   13866   8712   5743   3536   1753   4408   
LAW ENFORCEMENT RELATED    1156     929    715    437    293    125    110   
MISCELLANEOUS CRIME       30123   12142   9711   7291   5576   5302   7516   
NON-CRIMINAL                  1       3      1      0      0      0      0   
PROPERTY CRIME           263736  112344  96253  82190  66718  62126  76181   
PUBLIC ORDER               6102    4430   3484   2356   1413   1693   3793   
SEXUAL / TRAFFICKING      10428    2955   2961   2785   2163   1648   1459   
VIOLENT CRIME            109441   96703  85604  69120  50812  37683  34551   
WEAPONS OFFENSE            8475    5898   4355   3010   1948   1091   1116   

arrest_hour                  7       8       9       10      11

In [10]:
crime_rate = df.groupby("Primary Type")["Arrest"] \
               .mean() \
               .reset_index(name="arrest_rate")

print(crime_rate)

              Primary Type  arrest_rate
0  CRIMES AGAINST CHILDREN     0.184384
1             DRUG OFFENSE     0.992657
2  LAW ENFORCEMENT RELATED     0.919886
3      MISCELLANEOUS CRIME     0.176523
4             NON-CRIMINAL     0.360000
5           PROPERTY CRIME     0.117822
6             PUBLIC ORDER     0.863999
7     SEXUAL / TRAFFICKING     0.179913
8            VIOLENT CRIME     0.197139
9          WEAPONS OFFENSE     0.725496


In [11]:
domestic_counts = df.groupby(["Primary Type", "Domestic"]) \
                .size() \
                .unstack(fill_value=0)

domestic_counts["domestic_rate"] = (
    domestic_counts[True] /
    (domestic_counts[True] + domestic_counts[False])
)

print(domestic_counts)


Domestic                   False    True  domestic_rate
Primary Type                                           
CRIMES AGAINST CHILDREN    14190   44199       0.756975
DRUG OFFENSE              697906     600       0.000859
LAW ENFORCEMENT RELATED    19799     185       0.009257
MISCELLANEOUS CRIME       307668  182613       0.372466
NON-CRIMINAL                  24       1       0.040000
PROPERTY CRIME           3785507  192245       0.048330
PUBLIC ORDER              139422    2790       0.019619
SEXUAL / TRAFFICKING       55970   13469       0.193969
VIOLENT CRIME            1351966  930327       0.407628
WEAPONS OFFENSE           121563     745       0.006091


In [12]:
end = time.time() #How long it takes to process and pipeline the data

print("Execution time:", round(end - start, 2), "seconds") #Going to be less with smaller dataset

Execution time: 111.0 seconds


In [13]:
ram = df.memory_usage(deep=True).sum() / 1024**2 #How much RAM Dataframe is using

print("Memory Usage:", round(ram, 2), "MB")

Memory Usage: 4633.41 MB
